### Import all necessary modules and libraries

In [ ]:
from agent.agent import BaseAgent
from env.replay import Backtest

import numpy as np
import pandas as pd

### Define your agent by inheriting from BaseAgent

1. Define the class and the constructor
2. Implement the event methods: on_quote, on_trade, on_time.
- In "on_quote", you can react upon the limit order submissions or cancellations in the market. In this example we do not react to any of these updates.
- In "on_trade", you can react upon the execution of trade in the market. In this example we react by randomly submitting buy or sell limit orders at the best bid or ask price, respectively.
- In "on_time", you can react upon the passage of time. In this example we do not react to the passage of time.

In [ ]:
class Agent(BaseAgent):

    def __init__(self, name, p1, p2, *args, **kwargs):
        """
        :param name:
            str, agent name
        :param p1:
            float, probability of limit order submission
        :param p2:
            float, cond. probability of buy limit order submission
        """
        super(Agent, self).__init__(name, *args, **kwargs)
        self.p1 = p1
        self.p2 = p2

    def on_quote(self, market_id:str, book_state:pd.Series):
        """
        This method is called after a new quote.

        :param market_id:
            str, market identifier
        :param book_state:
            pd.Series, including timestamp, bid/ask price/quantity for 10 levels
        """
        pass

    def on_trade(self, market_id:str, trades_state:pd.Series):
        """
        This method is called after a new trade.

        :param market_id:
            str, market identifier
        :param trades_state:
            pd.Series, including timestamp, price, quantity
        """

        # continuously uniformly distributed random variable z1
        z1 = np.random.rand()

        if z1 < self.p1:

            # continuously uniformly distributed random variable z2
            z2 = np.random.rand()

            if z2 < self.p2:
                self.market_interface.submit_order(
                    market_id, 'buy', 100, self.market_interface.market_state_list[market_id].best_bid)
            else:
                self.market_interface.submit_order(
                    market_id, 'sell', 100, self.market_interface.market_state_list[market_id].best_ask)

    def on_time(self, timestamp:pd.Timestamp, timestamp_next:pd.Timestamp):
        """
        This method is called with every iteration and provides the timestamps
        for both current and next iteration. The given interval may be used to
        submit orders before a specific point in time.

        :param timestamp:
            pd.Timestamp, timestamp recorded
        :param timestamp_next:
            pd.Timestamp, timestamp recorded in next iteration
        """

        pass

### Define your agent and backtest

1. Define an instance of your agent by specifying the parameters defined in the constructor.
2. Define an instance of the Backtest class by passing your agent instance.
3. Define the list of asset identifiers you want to trade on.

In [ ]:
agent = Agent(
    name="intro_agent",
    p1=0.2,
    p2=0.75,
    exposure_limit=1_000_000,  # exposure limit of EUR 1 million
    latency=10,  # latency in us (microseconds)
    transaction_cost_factor=5e-5,  # 0.5 bps variable transaction cost
)

backtest = Backtest(
    agent=agent,
)

identifier_list = [
    # Aktie1
    "Aktie1.BOOK", "Aktie1.TRADES",
]

### Run your backtest

There are several methods to run your backtest. Here, we show two options:

1. Run your agent within predefined time periods (called "episodes") using the method **run_episode_list**. The episode list requires three date-time strings per episode: the first defines the start of the episode buffer. The simulation starts here to fill the order book based on historical order flow. The second timestamp is the start of the backtest where your agent can interact with the market. The third timestamp represents the end of the backtest.
2. Run your agent against a series of randomly selected episodes using the method **run_episode_generator**. The backtest periods are selected within the given date range between date_start and date_end. The episode_interval dividies each day into intervals of the specified length (in minutes). From these intervals, episodes are randomly selected. Each episode consists of: buffer period defined by episode_buffer (in minutes) and backtest period defined by episode_length (in minutes). Thus, the buffer and length of each episode should sum up to the episode_interval. To ensure reproducibility, you can set a random seed. You can also specify the number of episodes to run. If episode_shuffle is set to True, the episodes are randomly shuffled before running the backtest.

Furthermore, both methods require the source directory where the historical market data is stored. Ultimately, you can define the sampling frequency (in events) to define how often your agent can interact with the market. By default, this is set to 1, meaning that your agent can react to every market event (quote or trade). Higher values imply that your agent can only react every n-th event, but this can speed up the backtesting process.

In [ ]:
# Option 1: run agent against a single episode defined by start and end date
backtest.run_episode_list(
    identifier_list=identifier_list,
    source_directory=r'ITFM_Data',
    episode_list=[
        ("2025-03-17T08:00:00", "2025-03-17T08:05:00", "2025-03-17T08:20:00"),
        ("2025-03-17T09:00:00", "2025-03-17T09:05:00", "2025-03-17T09:20:00"),
        # ...
    ],
)

To keep the runtime of this example short, the second option is commented out.

In [ ]:
# Option 2: run agent against a series of generated episodes, that is,
# generate episodes with the same episode_buffer and episode_length
#backtest.run_episode_generator(
#    identifier_list=identifier_list,
#    source_directory=r"ITFM_Data",
#    date_start="2025-03-17",
#    date_end="2025-03-17",
#    episode_interval=20,
#    episode_shuffle=False,
#    episode_buffer=5,
#    episode_length=15,
#    num_episodes=2,
#    seed=123,
#    sampling_freq=5,
#)

### Evaluate your backtesting results

You can access the backtesting results via the attribute **results** of your backtest instance. This attribute is a list of dictionaries, where each dictionary contains various a list of orders, trades and performance metrics for each episode in the backtest. You can analyze these results to evaluate the performance of your agent.

In [ ]:
# TODO: EVALUATE YOUR BACKTESTING RESULTS
# E.g., you can use result list backtest.results to analyze your runs
pnl_realized_all_runs = [d['PnL_realized'] for d in backtest.results]
pnl_realized_all_runs = pd.DataFrame(pnl_realized_all_runs)
pnl_realized_all_runs.to_csv('PnL_realized_all_runs.csv', index_label='Run')